# TeleSale ASR trên Colab (whisper-turbo / large-v3 / PhoWhisper, pipeline B2)
Runtime → Change runtime type → **T4 GPU**, rồi chạy lần lượt cell 1 → 6.
Muốn đổi model: sửa `MODELS` ở cell 3 rồi chạy lại cell 3 → 4.

In [ ]:
# 1. Lấy code
import subprocess
subprocess.run('test -d /content/repo || git clone --depth 1 https://github.com/PhucNguyenThanh04/TeleSale-SudoCode2026.git /content/repo',
               shell=True, check=True)
%cd /content/repo

In [ ]:
# 2. Thư viện + CUDA 12 cho faster-whisper (Colab mới không có sẵn libcublas.so.12)
!pip install -q -r speech/requirements.txt fastapi "uvicorn[standard]" python-multipart onnxruntime "av==18.1.0" nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile /content/repo/asr_server.py
# 3. Server ASR: FastAPI bọc speech.asr (B2: VAD -> Whisper + hotword -> lexicon/ITN -> entity -> turns[])
import ctypes, glob, threading, time
from datetime import date
from pathlib import Path

# Nạp sẵn cuBLAS/cuDNN 12 từ pip để CTranslate2 tìm thấy (không cần LD_LIBRARY_PATH)
for pattern in ("libcublasLt.so.12", "libcublas.so.12", "libcudnn*.so.9"):
    for lib in sorted(glob.glob(f"/usr/local/lib/python3*/dist-packages/nvidia/*/lib/{pattern}")):
        ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)

from fastapi import FastAPI, File, HTTPException, Query, UploadFile
from speech import asr, audio as au
from speech.btc import load_catalog
from speech.normalize import default_lexicon

# ---- Cấu hình: sửa trực tiếp ở đây ----
MODELS = ["whisper-turbo", "whisper-large-v3", "phowhisper-medium"]  # phowhisper-large: gọi với hotwords=false
DEVICE, COMPUTE_TYPE = "cuda", "float16"
BTC_DIR = Path("/content/drive/MyDrive/BTC-Data-Vong1-TEAMS")  # không có thì entity sku để trống
# ----------------------------------------

CATALOG = load_catalog(BTC_DIR if (BTC_DIR / "catalog" / "products.json").is_file() else None)
LEXICON = default_lexicon()
_models, _locks, _load_lock = {}, {}, threading.Lock()


def get_model(name: str):
    if name not in MODELS:
        raise HTTPException(400, f"Model chưa bật: {name}. Đang bật: {MODELS}")
    with _load_lock:
        if name not in _models:
            _models[name] = asr.load_model(name, DEVICE, COMPUTE_TYPE)[0]
            _locks[name] = threading.Lock()
    return _models[name], _locks[name]


app = FastAPI(title="TeleSale ASR (Colab)")


@app.on_event("startup")
def warmup():
    for name in MODELS:
        get_model(name)


@app.get("/health")
def health():
    return {"loaded": sorted(_models), "models": MODELS, "catalog_skus": len(CATALOG)}


@app.post("/v1/asr/transcribe")
def transcribe(file: UploadFile = File(...), model: str = Query(MODELS[0]),
               hotwords: bool = Query(True), call_date: str | None = Query(None)):
    m, lock = get_model(model)
    audio = au.decode(file.file.read())
    ref = date.fromisoformat(call_date) if call_date else asr.REFERENCE_DATE
    t0 = time.perf_counter()
    with lock:
        out = asr.transcribe_audio(m, audio, cfg=asr.DecodeConfig(use_hotwords=hotwords),
                                   vad=asr.VadConfig(), lexicon=LEXICON,
                                   catalog=CATALOG, reference_date=ref)
    sec = time.perf_counter() - t0
    dur = audio.size / au.SAMPLE_RATE
    out["meta"] = {"model": model, "audio_s": round(dur, 2), "asr_s": round(sec, 2),
                   "asr_s_per_audio_min": round(60 * sec / dur, 1) if dur else None}
    return out

In [ ]:
# 4. Khởi động server (tắt server cũ nếu có). Lần đầu tải model mất vài phút.
# Muốn dùng catalog BTC: mount Drive trước (from google.colab import drive; drive.mount('/content/drive')).
import time, requests
subprocess.run(['pkill', '-f', 'uvicorn asr_server'])
server = subprocess.Popen(['python', '-m', 'uvicorn', 'asr_server:app', '--host', '127.0.0.1', '--port', '8000'],
                          cwd='/content/repo', start_new_session=True,
                          stdout=open('/content/asr.log', 'w'), stderr=subprocess.STDOUT)
ok = False
for _ in range(120):
    if server.poll() is not None:
        break  # server đã thoát vì lỗi
    try:
        print(requests.get('http://127.0.0.1:8000/health', timeout=3).json())
        ok = True
        break
    except Exception:
        time.sleep(5)
if not ok:
    print('SERVER LỖI, log cuối:')
    print(open('/content/asr.log').read()[-3000:])

In [ ]:
# 5. Mở ra internet bằng Cloudflare quick tunnel (chỉ một tunnel)
import os, re
if not os.path.exists('/content/cloudflared'):
    subprocess.run(['curl', '-fsSL', '--retry', '3', '--max-time', '120', '-o', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    os.chmod('/content/cloudflared', 0o755)
subprocess.run(['pkill', '-f', 'cloudflared tunnel'])
time.sleep(1)
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                          start_new_session=True, stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(30):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        PUBLIC_URL = m.group(0)
        break
    time.sleep(2)
print(PUBLIC_URL or open('/content/tunnel.log').read()[-2000:])

In [ ]:
# 6. Kiểm tra qua link public: health + gửi thử 1 file audio (chọn file .wav/.mp3 từ máy)
time.sleep(5)  # chờ DNS của tunnel
print(requests.get(f'{PUBLIC_URL}/health', timeout=30).json())
from google.colab import files
for name, data in files.upload().items():
    r = requests.post(f'{PUBLIC_URL}/v1/asr/transcribe', params={'model': 'whisper-turbo', 'call_date': '2026-10-15'},
                      files={'file': (name, data)}, timeout=600)
    r.raise_for_status()
    out = r.json()
    print(out['meta'])
    print(out['text'][:500])
    print(out['entities'])

# TeleSale ASR API

Base URL: `PUBLIC_URL` in ra ở cell 5 (dạng `https://xxxx.trycloudflare.com`). Link đổi mỗi lần chạy lại cell 5; phiên Colab sống khoảng 90 phút.
Swagger UI để thử trực tiếp trên trình duyệt: `PUBLIC_URL/docs`

## 1. `GET /health`
Kiểm tra server và các model đã nạp.

| Trường | Kiểu | Ý nghĩa |
|---|---|---|
| `loaded` | list | Model đã nạp lên GPU |
| `models` | list | Model được phép gọi |
| `catalog_skus` | int | Số SKU trong catalog BTC (0 = chưa mount Drive, không trích được `sku`) |

## 2. `POST /v1/asr/transcribe`
Gửi 1 file audio, nhận transcript, entity và lượt nói.

**Body**: `multipart/form-data`, field `file` = file audio (wav/mp3/m4a…, mono hoặc stereo, tự resample về 16 kHz).

**Query params**

| Tham số | Mặc định | Ý nghĩa |
|---|---|---|
| `model` | `whisper-turbo` | Một trong `MODELS` ở cell 3 |
| `hotwords` | `true` | `true` = profile **B2** (khuyên dùng), `false` = **B1**. `phowhisper-large` phải dùng `false` |
| `call_date` | `2026-10-15` | Ngày cuộc gọi `YYYY-MM-DD`, dùng để quy đổi "mai", "thứ bảy"… thành ngày cụ thể |

**Response**

| Trường | Ý nghĩa |
|---|---|
| `text` | Transcript cả cuộc gọi (dạng nói, dùng chấm WER) |
| `entities` | Entity cấp hội thoại: `sku`, `price_vnd`, `phone`, `order_id`, `payment`, `delivery_days`, `promo_expiry`… |
| `turns[]` | Từng lượt: `idx`, `speaker` (`A` = nhân viên, `C` = khách), `start`, `end`, `text`, `entities` |
| `asr_segments[]` | Đoạn VAD gốc (debug): `raw`, `avg_logprob`, `no_speech_prob`, `speaker` |
| `meta` | `model`, `audio_s`, `asr_s`, `asr_s_per_audio_min` (số giây xử lý cho mỗi phút audio) |

**Ví dụ (PowerShell)**

    curl.exe -X POST "PUBLIC_URL/v1/asr/transcribe?model=whisper-turbo&call_date=2026-10-15" -F "file=@D01.wav"

**Ví dụ (Python)**

    import requests
    r = requests.post(f"{PUBLIC_URL}/v1/asr/transcribe",
                      params={"model": "whisper-turbo", "call_date": "2026-10-15"},
                      files={"file": open("D01.wav", "rb")}, timeout=600)
    out = r.json(); print(out["text"]); print(out["entities"])

**Lỗi thường gặp**

| Mã | Nguyên nhân |
|---|---|
| `404 Not Found` | Gọi sai đường dẫn (vd. mở `/` thay vì `/docs`) |
| `400 Model chưa bật` | `model` không có trong `MODELS` ở cell 3 |
| `422` | Thiếu field `file` hoặc `call_date` sai định dạng |
| `502` / `530` | Server ở cell 4 đã chết hoặc tunnel hết hạn; chạy lại cell 4–5 |

## 3. Kết quả tham khảo (whisper-turbo, B2)

| Bộ audio | WER | CER | Entity | Entity theo lượt | Tách người nói |
|---|---|---|---|---|---|
| team22 (22 file) | 13,7% | 9,6% | 73,4% | 60,6% | 88,9% |
| btc-edge (D01–D04) | 7,0% | 3,5% | 93,1% | 85,7% | 91,2% |

Số đo trên RTX 3050 (chi tiết: `speech/report.md`). T4 nhanh hơn, nên `asr_s_per_audio_min` sẽ thấp hơn số trong báo cáo.

Server không có xác thực; chỉ gửi audio TTS hoặc audio đã được đồng ý, không gửi ghi âm khách hàng thật.